# CNN VAE for implied volatility surfaces

This notebook isolates the **convolutional encoder** experiment from [`vae_volatility_surface.ipynb`](vae_volatility_surface.ipynb): unconstrained conv decoder vs **structural strike no-arbitrage** decoder (row-wise call prices → Black-76 IV), on the same 8×5 (**tenor × delta**) grid and mixed **SABR + SSVI** training data.

**Dependencies:** same as the main notebook (`pip install -e ".[vae]"`). Open the repo root (or `pip install -e .`) so `helper_module` resolves.


In [2]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _base in [_root, *_root.parents]:
    if (_base / "helper_module").is_dir() and (_base / "pyproject.toml").is_file():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break
else:
    raise RuntimeError("Open from repo root or pip install -e .")

import importlib

import matplotlib.pyplot as plt
import numpy as np
import helper_module.vae_surface
import helper_module.vae_vol_surface
import torch
from helper_module.arbitrage import validate_vol_surface_per_expiry_black76

# Ensure notebook sees latest symbols after module edits.
importlib.reload(helper_module.vae_vol_surface)
importlib.reload(helper_module.vae_surface)

from helper_module.vae_surface import (
    DELTAS,
    TENORS_YEARS,
    make_synthetic_sabr_surfaces,
    make_synthetic_ssvi_surfaces,
    strikes_for_surface_grid,
)

plt.rcParams.update({"font.sans-serif": ["DejaVu Sans", "Arial", "sans-serif"]})

SABR_MIX_WEIGHT = 0.5  # fraction of SABR in the mixed SABR+SSVI calibration dataset

## 1. CNN encoder-decoder (stepwise calibration on mixed SABR + SSVI)

Two **calibration** passes share the same synthetic mix **`surfaces_cnn`**:

- **Step 1 — `cnn_unconstrained`:** conv decoder → softplus vols.
- **Step 2 — `cnn_structural`:** same encoder template, structural strike decoder (monotone/convex calls → σ).
- **Step 3:** sampling checker stats and training-loss curves for the models trained in Steps 1–2.

Calendar / sticky-delta is **not** hard-coded (same story as the MLP constrained VAE).


### Architecture and helpers

`ConvSurfaceVAE` defines the CNN VAE; `train_conv_vae` runs calibration on a NumPy batch; `arbitrage_pass_rate` scores random decodes with the discrete Black–76 checker.

**Next:** Step 1 builds **`surfaces_cnn`** (mixed SABR+SSVI) and fits **`cnn_unconstrained`**. Step 2 re-trains on the **same** data with the structural decoder → **`cnn_structural`**. Step 3 compares those two calibrated models.


In [3]:
from torch import nn
import helper_module.vae_vol_surface as vvs


class ConvSurfaceVAE(nn.Module):
    def __init__(
        self,
        latent_dim: int = 8,
        constrained_strike_decoder: bool = False,
        forward_rate: float = 0.03,
        row_hidden: int = 32,
    ):
        super().__init__()
        self.latent_dim = latent_dim
        self.constrained_strike_decoder = bool(constrained_strike_decoder)
        # Must not be named `forward`: it shadows nn.Module.forward().
        self.forward_rate = float(forward_rate)
        n_t, n_d = len(TENORS_YEARS), len(DELTAS)

        self.enc = nn.Sequential(
            nn.Conv2d(1, 12, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(12, 24, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(24 * n_t * n_d, 128),
            nn.ReLU(),
        )
        self.mu_head = nn.Linear(128, latent_dim)
        self.logvar_head = nn.Linear(128, latent_dim)

        self.register_buffer(
            "K_grid",
            torch.tensor(strikes_for_surface_grid(self.forward_rate), dtype=torch.float32),
        )

        if self.constrained_strike_decoder:
            h = row_hidden
            self.row_net = nn.Sequential(
                nn.Linear(latent_dim + 1, h),
                nn.ReLU(),
                nn.Linear(h, h),
                nn.ReLU(),
                nn.Linear(h, 5),
            )
            self.register_buffer(
                "T_row",
                torch.tensor(TENORS_YEARS, dtype=torch.float32).view(n_t, 1),
            )
        else:
            self.dec_fc = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.ReLU(),
                nn.Linear(128, 24 * n_t * n_d),
                nn.ReLU(),
            )
            self.dec_conv = nn.Sequential(
                nn.Conv2d(24, 12, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv2d(12, 1, kernel_size=3, padding=1),
            )

    def encode(self, x_flat: torch.Tensor):
        x = x_flat.view(-1, 1, len(TENORS_YEARS), len(DELTAS))
        h = self.enc(x)
        return self.mu_head(h), self.logvar_head(h)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        n_t, n_d = len(TENORS_YEARS), len(DELTAS)
        B = z.shape[0]
        if self.constrained_strike_decoder:
            log_t = torch.log1p(self.T_row).view(n_t)
            F = z.new_tensor(self.forward_rate)
            D = torch.ones((), device=z.device, dtype=z.dtype)
            fw = float(self.forward_rate)
            rows: list[torch.Tensor] = []
            for i in range(n_t):
                inp = torch.cat([z, log_t[i].expand(B, 1)], dim=-1)
                raw = self.row_net(inp)
                C_row = vvs.call_prices_monotone_convex_five_strikes(
                    raw, forward=fw, discount=1.0
                )
                K_row = self.K_grid[i : i + 1, :].expand(B, -1)
                T_row = self.T_row[i : i + 1, :].expand(B, -1)
                sig_row = vvs.black76_implied_vol_newton_torch(C_row, F, K_row, T_row, D)
                rows.append(sig_row)
            return torch.stack(rows, dim=1).reshape(B, n_t * n_d)

        h = self.dec_fc(z)
        h = h.view(-1, 24, n_t, n_d)
        out = self.dec_conv(h)
        vol = torch.nn.functional.softplus(out) + 1e-4
        return vol.view(-1, n_t * n_d)

    def forward(self, x_flat: torch.Tensor):
        mu, logvar = self.encode(x_flat)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        recon = self.decode(z)
        return recon, mu, logvar


def train_conv_vae(
    surfaces_np: np.ndarray,
    *,
    latent_dim: int = 8,
    constrained_strike_decoder: bool = False,
    epochs: int = 30,
    batch_size: int = 128,
    lr: float = 8e-4,
    kl_weight: float = 0.25,
):
    device = torch.device(vvs.pick_torch_training_device())
    print(f"train_conv_vae: {device}")
    x = torch.tensor(surfaces_np, dtype=torch.float32, device=device)
    model = ConvSurfaceVAE(
        latent_dim=latent_dim,
        constrained_strike_decoder=constrained_strike_decoder,
        forward_rate=0.03,
    ).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    losses = []
    model.train()
    n = x.shape[0]
    for _ in range(epochs):
        perm = torch.randperm(n, device=device)
        ep = 0.0
        steps = 0
        for s in range(0, n, batch_size):
            b = x[perm[s : s + batch_size]]
            opt.zero_grad(set_to_none=True)
            recon, mu, logvar = model(b)
            mse = torch.nn.functional.mse_loss(recon, b)
            kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = mse + kl_weight * kl
            loss.backward()
            opt.step()
            ep += float(loss.item())
            steps += 1
        losses.append(ep / max(steps, 1))
    return model, losses


def arbitrage_pass_rate(model: nn.Module, n_eval: int = 800, forward: float = 0.03):
    model.eval()
    dev = next(model.parameters()).device
    K = strikes_for_surface_grid(forward)
    n_t, n_d = len(TENORS_YEARS), len(DELTAS)

    with torch.no_grad():
        z = torch.randn(n_eval, model.latent_dim, device=dev)
        g = model.decode(z).cpu().numpy().reshape(n_eval, n_t, n_d)

    n_ok = 0
    n_strike = 0
    n_sticky = 0
    n_both = 0
    for i in range(n_eval):
        rep = validate_vol_surface_per_expiry_black76(K, TENORS_YEARS, g[i], forward=forward)
        has_strike = any("Sticky-delta" not in m for m in rep.violations)
        has_sticky = any("Sticky-delta" in m for m in rep.violations)
        n_ok += int(rep.ok)
        n_strike += int(has_strike)
        n_sticky += int(has_sticky)
        n_both += int(has_strike and has_sticky)
    return {
        "pass_rate": n_ok / n_eval,
        "n_ok": n_ok,
        "n_eval": n_eval,
        "n_strike": n_strike,
        "n_sticky": n_sticky,
        "n_both": n_both,
    }




### Step 1 — Calibrate CNN on mixed SABR + SSVI (unconstrained decoder)

Stack synthetic **SABR** and **SSVI** rows (`SABR_MIX_WEIGHT`), shuffle into **`surfaces_cnn`**, then calibrate **`cnn_unconstrained`**. After training, the cell shows:

- **2×2 heatmaps:** one fresh **SABR** and one fresh **SSVI** surface vs **deterministic** CNN reconstructions (encode mean $\mu$, decode).
- **Training loss vs epoch** for Step 1 (`loss_u`), with raw curve, short moving average, and best-epoch marker.
- **RMSE** on each sample grid (absolute; and as **% of mean $|\sigma|$**), plus a pooled average.

Later steps reuse **`cnn_unconstrained`** and **`loss_u`**.


In [4]:
# Mixed classical dataset for CNN experiment (SABR + SSVI model averaging).
rng = np.random.default_rng(0)
n_train_cnn = 3000
n_sabr_cnn = int(round(n_train_cnn * SABR_MIX_WEIGHT))
n_ssvi_cnn = n_train_cnn - n_sabr_cnn
surfaces_cnn = np.vstack([
    make_synthetic_sabr_surfaces(n_sabr_cnn, rng=rng),
    make_synthetic_ssvi_surfaces(n_ssvi_cnn, rng=rng),
])
rng.shuffle(surfaces_cnn, axis=0)

cnn_unconstrained, loss_u = train_conv_vae(
    surfaces_cnn,
    latent_dim=10,
    constrained_strike_decoder=False,
    epochs=35,
    batch_size=128,
    lr=8e-4,
    kl_weight=0.25,
)

# --- Step 1 diagnostics: SABR vs SSVI recon, training loss, RMSE ---
n_t, n_d = len(TENORS_YEARS), len(DELTAS)
cnn_unconstrained.eval()
_dev = next(cnn_unconstrained.parameters()).device

sabr_sample = make_synthetic_sabr_surfaces(1, rng=np.random.default_rng(42)).reshape(n_t, n_d)
ssvi_sample = make_synthetic_ssvi_surfaces(1, rng=np.random.default_rng(43)).reshape(n_t, n_d)


def reconstruct_surface_with_mean_latent(mat: np.ndarray) -> np.ndarray:
    x = torch.tensor(mat.reshape(1, -1), dtype=torch.float32, device=_dev)
    with torch.no_grad():
        mu, _ = cnn_unconstrained.encode(x)
        out = cnn_unconstrained.decode(mu)
    return out.cpu().numpy().reshape(n_t, n_d)


recon_sabr = reconstruct_surface_with_mean_latent(sabr_sample)
recon_ssvi = reconstruct_surface_with_mean_latent(ssvi_sample)

_eps = 1e-12


def report_rmse(label: str, truth: np.ndarray, recon: np.ndarray) -> tuple[float, float]:
    diff = truth - recon
    rmse = float(np.sqrt(np.mean(diff**2)))
    scale = float(np.mean(np.abs(truth)))
    rel_pct = 100.0 * rmse / max(scale, _eps)
    print(f"{label}: RMSE = {rmse:.6f}  |  RMSE / mean(|σ|) = {rel_pct:.3f}%")
    return rmse, rel_pct


rmse_sabr, rel_sabr = report_rmse("SABR sample vs CNN recon", sabr_sample, recon_sabr)
rmse_ssvi, rel_ssvi = report_rmse("SSVI sample vs CNN recon", ssvi_sample, recon_ssvi)
rmse_mean = 0.5 * (rmse_sabr + rmse_ssvi)
rel_mean = 0.5 * (rel_sabr + rel_ssvi)
print(f"Pooled (mean of two surfaces): RMSE = {rmse_mean:.6f}  |  relative % (avg) = {rel_mean:.3f}%")

_vmin = float(min(sabr_sample.min(), ssvi_sample.min(), recon_sabr.min(), recon_ssvi.min()))
_vmax = float(max(sabr_sample.max(), ssvi_sample.max(), recon_sabr.max(), recon_ssvi.max()))

fig_surf, axes_surf = plt.subplots(2, 2, figsize=(10, 7), dpi=130, constrained_layout=True)
for ax, title, mat in zip(
    axes_surf.ravel(),
    ("SABR sample σ", "CNN recon (μ decode)", "SSVI sample σ", "CNN recon (μ decode)"),
    (sabr_sample, recon_sabr, ssvi_sample, recon_ssvi),
):
    im_s = ax.imshow(mat, origin="lower", interpolation="nearest", cmap="viridis", aspect="auto", vmin=_vmin, vmax=_vmax)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Δ bucket")
    ax.set_ylabel("Tenor bucket")
fig_surf.suptitle(
    "Step 1: one SABR & one SSVI surface vs calibrated CNN (deterministic μ decode)",
    fontsize=12,
    weight="bold",
)
fig_surf.colorbar(im_s, ax=axes_surf, shrink=0.75, label=r"$\sigma$")
plt.show()

# Loss vs epoch (Step 1 calibration only)
loss_u_arr = np.asarray(loss_u, dtype=float)
epochs_u = np.arange(1, len(loss_u_arr) + 1)
win_u = min(5, len(loss_u_arr))
if win_u >= 2:
    kernel_u = np.ones(win_u) / win_u
    smooth_u = np.convolve(loss_u_arr, kernel_u, mode="same")
else:
    smooth_u = loss_u_arr

fig_loss, ax_loss = plt.subplots(figsize=(8, 4.6), dpi=130)
ax_loss.set_facecolor("#f7f9fc")
ax_loss.plot(epochs_u, loss_u_arr, color="#f58518", lw=1.8, alpha=0.35, label="Raw epoch loss")
ax_loss.plot(epochs_u, smooth_u, color="#f58518", lw=2.8, label=f"Moving avg (window={win_u})")
ax_loss.set_title("Step 1 — CNN unconstrained training loss vs epoch", fontsize=12, weight="bold")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("ELBO proxy loss")
ax_loss.grid(True, alpha=0.25, linestyle="--", linewidth=0.8)
ax_loss.spines[["top", "right"]].set_visible(False)
ax_loss.legend(frameon=False, fontsize=9)
_best_ep = int(np.argmin(loss_u_arr)) + 1
ax_loss.scatter([_best_ep], [float(np.min(loss_u_arr))], color="#b85e00", s=28, zorder=5)
plt.tight_layout()
plt.show()

_final_loss = float(loss_u_arr[-1])
print(f"Final epoch loss (Step 1): {_final_loss:.6f}  (best epoch {_best_ep}, min {float(np.min(loss_u_arr)):.6f})")


train_conv_vae: mps


### Step 2 — Same mixed data, structural strike-no-arb decoder

Reuse **`surfaces_cnn`** from Step 1. Train **`cnn_structural`** via the same `train_conv_vae` helper with `constrained_strike_decoder=True` (row-wise calls → IV). Both **`cnn_unconstrained`** and **`cnn_structural`** share the `ConvSurfaceVAE` class; only the decoder branch differs.


In [ ]:
cnn_structural, loss_b = train_conv_vae(
    surfaces_cnn,
    latent_dim=10,
    constrained_strike_decoder=True,
    epochs=35,
    batch_size=128,
    lr=8e-4,
    kl_weight=0.25,
)



train_conv_vae: mps


### Step 3 — Diagnostics vs Step 1 / Step 2 models

Evaluate **`cnn_unconstrained`** and **`cnn_structural`** with `arbitrage_pass_rate`, then plot **`loss_u`** and **`loss_b`** from the calibrations above.


In [ ]:
rep_u = arbitrage_pass_rate(cnn_unconstrained, n_eval=1000)
rep_b = arbitrage_pass_rate(cnn_structural, n_eval=1000)

print("CNN unconstrained (conv decoder → σ):")
print(f"  arbitrage-free: {rep_u['n_ok']}/{rep_u['n_eval']} = {rep_u['pass_rate']:.1%}")
print(f"  strike arb: {rep_u['n_strike']}/{rep_u['n_eval']} | sticky arb: {rep_u['n_sticky']}/{rep_u['n_eval']} | both: {rep_u['n_both']}/{rep_u['n_eval']}")
print("CNN structural strike-no-arb (row MLP → monotone/convex calls → σ):")
print(f"  arbitrage-free: {rep_b['n_ok']}/{rep_b['n_eval']} = {rep_b['pass_rate']:.1%}")
print(f"  strike arb: {rep_b['n_strike']}/{rep_b['n_eval']} | sticky arb: {rep_b['n_sticky']}/{rep_b['n_eval']} | both: {rep_b['n_both']}/{rep_b['n_eval']}")

epochs_cnn = np.arange(1, len(loss_u) + 1)
loss_u_arr = np.asarray(loss_u, dtype=float)
loss_b_arr = np.asarray(loss_b, dtype=float)
win_cnn = min(5, len(loss_u_arr))
if win_cnn >= 2:
    kernel_cnn = np.ones(win_cnn) / win_cnn
    smooth_u = np.convolve(loss_u_arr, kernel_cnn, mode="same")
    smooth_b = np.convolve(loss_b_arr, kernel_cnn, mode="same")
else:
    smooth_u = loss_u_arr
    smooth_b = loss_b_arr

fig, ax = plt.subplots(figsize=(8, 4.8), dpi=130)
ax.set_facecolor("#f7f9fc")

ax.plot(epochs_cnn, loss_u_arr, color="#f58518", lw=1.8, alpha=0.28)
ax.plot(epochs_cnn, smooth_u, color="#f58518", lw=2.8, label="CNN unconstrained")
ax.plot(epochs_cnn, loss_b_arr, color="#54a24b", lw=1.8, alpha=0.28)
ax.plot(epochs_cnn, smooth_b, color="#2ca02c", lw=2.8, label="CNN structural strike-no-arb")

ax.set_title("CNN-VAE loss: unconstrained conv decoder vs structural strike decoder", fontsize=12.5, weight="bold")
ax.set_xlabel("Epoch", fontsize=10.5)
ax.set_ylabel("ELBO proxy loss", fontsize=10.5)
ax.grid(True, alpha=0.25, linestyle="--", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=9)

best_u_ep = int(np.argmin(loss_u_arr)) + 1
best_b_ep = int(np.argmin(loss_b_arr)) + 1
ax.scatter([best_u_ep], [float(np.min(loss_u_arr))], color="#b85e00", s=28, zorder=5)
ax.scatter([best_b_ep], [float(np.min(loss_b_arr))], color="#1a7f1a", s=28, zorder=5)

plt.tight_layout()
plt.show()